# Stage 1 Checkpoint Evaluation

Evaluation-only extension of notebook 6. This compares saved Stage 1A and Stage 1B autoencoder checkpoints on fixed held-out test splits and writes the requested selection tables, per-example metrics, split fingerprints, plots, and final Stage 2 checkpoint manifest.

No training occurs in this notebook.

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())

# Match notebook 6's Colab setup: code repo in /content/neurovlm_gnn, Drive for data/run outputs.
REPO_URL = os.environ.get("NEUROVLM_REPO_URL", "https://github.com/neurovlm/neurovlm.git")
REPO_BRANCH = os.environ.get("NEUROVLM_REPO_BRANCH", "neurovlm_gnn")
REPO_DIR = Path(os.environ.get("NEUROVLM_REPO_DIR", "/content/neurovlm_gnn"))
DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", "/content/drive/MyDrive/neurovlm"))
INSTALL_DEPENDENCIES = os.environ.get("NEUROVLM_INSTALL_DEPS", "1") == "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    # Local fallback for running inside this workspace.
    local_repo = Path.cwd()
    if (local_repo / "experiments/3dcnn").exists():
        REPO_DIR = local_repo
        DRIVE_ROOT = Path(os.environ.get("NEUROVLM_DRIVE_ROOT", str(local_repo)))


def run_cmd(cmd, cwd=None, *, check=True):
    print("$", " ".join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        if check:
            raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

if IN_COLAB:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        run_cmd(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
    else:
        if not (REPO_DIR / ".git").exists():
            raise RuntimeError(
                f"{REPO_DIR} exists but is not a git checkout. Set NEUROVLM_REPO_DIR to a clean path "
                "or remove that folder, then rerun this cell."
            )
        run_cmd(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
        checkout = run_cmd(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=False)
        if checkout.returncode != 0:
            run_cmd(["git", "-C", str(REPO_DIR), "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"])
        run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])

os.chdir(REPO_DIR)

if INSTALL_DEPENDENCIES and IN_COLAB:
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "nilearn", "nibabel", "huggingface-hub", "safetensors", "adapters", "transformers", "pyarrow", "matplotlib", "pandas", "scikit-learn", "tqdm", "umap-learn"])
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-e", ".[viz,notebook,metrics]"])

sys.path.insert(0, str(REPO_DIR / "experiments" / "3dcnn"))
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR))

print("Working directory:", os.getcwd())
if (REPO_DIR / ".git").exists():
    print("Repo branch:", run_cmd(["git", "branch", "--show-current"], cwd=REPO_DIR, check=False).stdout.strip())
print("Drive root:", DRIVE_ROOT)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# The evaluator lives next to this notebook in experiments/3dcnn.
evaluator_path = REPO_DIR / "experiments/3dcnn/stage1_checkpoint_evaluation.py"
if not evaluator_path.exists():
    raise FileNotFoundError(
        f"Missing {evaluator_path}. Pull the version of the repo that includes "
        "experiments/3dcnn/stage1_checkpoint_evaluation.py, or upload that file next to this notebook."
    )

from stage1_checkpoint_evaluation import EvaluationConfig, run_evaluation

# Set this to the completed notebook-6 ablation run directory.
# Example: /content/drive/MyDrive/neurovlm/runs_atlas_free_cnn_ae_ablation/ae_ablation_20260623_123456
RUN_ROOT_VALUE = os.environ.get("NEUROVLM_AE_ABLATION_RUN_DIR", "").strip()
RUN_ROOT = Path(RUN_ROOT_VALUE).expanduser() if RUN_ROOT_VALUE else DRIVE_ROOT / "runs_atlas_free_cnn_ae_ablation/EDIT_ME_AE_ABLATION_RUN_DIR"
print("AE ablation run root:", RUN_ROOT)
if not RUN_ROOT_VALUE:
    print("Set NEUROVLM_AE_ABLATION_RUN_DIR or edit RUN_ROOT/run_dir entries before running evaluation.")

# If your actual paths differ, edit the run_dir strings directly here. Do not rely on mtime/order discovery.
AE_RUN_REGISTRY = {
    "mixed_baseline_raw_mse": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_baseline_raw_mse"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_balanced_raw_mse": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_balanced_raw_mse"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_balanced_hybrid_loss": {
        "run_dir": str(RUN_ROOT / "01_stage1_ae_pretraining/mixed_balanced_hybrid_loss"),
        "stage": "stage1a",
        "training_domain": "mixed",
        "test_domains": ["mixed", "pubmed", "nilearn", "neurovault"],
    },
    "mixed_to_pubmed": {
        "run_dir": str(RUN_ROOT / "02_stage1b_domain_finetune/pubmed/mixed_to_pubmed"),
        "stage": "stage1b",
        "training_domain": "pubmed",
        "test_domains": ["pubmed"],
        "cross_domain_test_domains": ["mixed", "nilearn", "neurovault"],
    },
    "mixed_to_nilearn": {
        "run_dir": str(RUN_ROOT / "02_stage1b_domain_finetune/nilearn/mixed_to_nilearn"),
        "stage": "stage1b",
        "training_domain": "nilearn",
        "test_domains": ["nilearn"],
        "cross_domain_test_domains": ["mixed", "pubmed", "neurovault"],
    },
    "mixed_to_neurovault": {
        "run_dir": str(RUN_ROOT / "02_stage1b_domain_finetune/neurovault/mixed_to_neurovault"),
        "stage": "stage1b",
        "training_domain": "neurovault",
        "test_domains": ["neurovault"],
        "cross_domain_test_domains": ["mixed", "pubmed", "nilearn"],
    },
}

# Notebook 6 used mixed_baseline_to_<domain> in STAGE1B_DOMAINS; prefer those dirs if they exist.
for key, domain in [("mixed_to_pubmed", "pubmed"), ("mixed_to_nilearn", "nilearn"), ("mixed_to_neurovault", "neurovault")]:
    alias_dir = RUN_ROOT / f"02_stage1b_domain_finetune/{domain}/mixed_baseline_to_{domain}"
    if alias_dir.exists():
        AE_RUN_REGISTRY[key]["run_dir"] = str(alias_dir)

TEST_JSONL = os.environ.get("NEUROVLM_TEST_JSONL", "")  # leave blank to use the repo fixed test split
OUTPUT_ROOT = Path(os.environ.get("NEUROVLM_AE_EVAL_OUTPUT_ROOT", DRIVE_ROOT / "runs_atlas_free_cnn_checkpoint_evaluation")).expanduser()

EVAL_BATCH_SIZE = int(os.environ.get("NEUROVLM_AE_EVAL_BATCH_SIZE", "32"))
EVAL_NUM_WORKERS = int(os.environ.get("NEUROVLM_EVAL_NUM_WORKERS", "2" if IN_COLAB else "0"))
OVERWRITE = False
MAKE_QUALITATIVE_PLOTS = True
EVALUATE_STAGE1B_CROSS_DOMAIN = True

AE_RUN_REGISTRY


In [ ]:
cfg = EvaluationConfig(
    registry=AE_RUN_REGISTRY,
    output_root=OUTPUT_ROOT,
    test_jsonl=Path(TEST_JSONL).expanduser() if TEST_JSONL else None,
    device="auto",
    eval_batch_size=EVAL_BATCH_SIZE,
    num_workers=EVAL_NUM_WORKERS,
    overwrite=OVERWRITE,
    make_qualitative_plots=MAKE_QUALITATIVE_PLOTS,
    evaluate_stage1b_cross_domain=EVALUATE_STAGE1B_CROSS_DOMAIN,
)

EVAL_OUTPUT_DIR = run_evaluation(cfg)
EVAL_OUTPUT_DIR

In [ ]:
import json
import pandas as pd

selected_path = EVAL_OUTPUT_DIR / "04_final_selection/selected_stage2_checkpoints.json"
manifest_path = EVAL_OUTPUT_DIR / "00_metadata/checkpoint_manifest.csv"
status_path = EVAL_OUTPUT_DIR / "00_metadata/run_status.json"
config_path = EVAL_OUTPUT_DIR / "00_metadata/evaluation_config.json"

with selected_path.open() as f:
    selected = json.load(f)
with status_path.open() as f:
    status = json.load(f)
with config_path.open() as f:
    eval_config = json.load(f)

print("Selected checkpoint manifest:", selected_path)
print("Configured run dirs:")
for name, spec in eval_config["registry"].items():
    run_dir = Path(spec["run_dir"])
    ckpt_dir = run_dir / "checkpoints" if (run_dir / "checkpoints").exists() else run_dir
    found = sorted(p.name for p in ckpt_dir.glob("*.pt")) if ckpt_dir.exists() else []
    print(f"- {name}: {run_dir}")
    print(f"  exists={run_dir.exists()} checkpoint_dir={ckpt_dir} pt_files={found[:12]}")

print("\nRun status summary:")
for key in ["variants_configured", "checkpoint_rows", "unique_model_states", "aliases_detected"]:
    print(f"- {key}: {status.get(key)}")
print(f"- load_failures: {len(status.get('load_failures', []))}")

if manifest_path.exists() and manifest_path.stat().st_size:
    manifest_df = pd.read_csv(manifest_path)
    display(manifest_df[[c for c in ["variant", "checkpoint_name", "checkpoint_path", "checkpoint_epoch", "load_status", "error_message"] if c in manifest_df.columns]])
else:
    print("\ncheckpoint_manifest.csv is empty. The configured run dirs did not contain the requested .pt checkpoint names.")

selected
